# Summary of Chemical Data Cleaning Work


The `DataCleaning.ipynb` notebook cleans the two raw and dirty chemical datasets,
`1991Che_DRSites.xls` and `log21P_MarjorChe_Sites.xlsx` stored in the `two_raw_datasets` folder. 

They contain the whole available chemical data, but are not in consistent formats and having some hidden issues within them.

The cleaning process is listed with these targeted issues in the following steps:

### Clean the log21 major chemical dataset

**Input**: the originally raw and dirty `log21P_MarjorChe_Sites.xlsx` dataset.

**Process**: 

Remove the rows with all missings and Retain needed columns $\rightarrow$ 

Back-transform the log2(Y+1) to the original Y values $\rightarrow$

Infer the units of all 16 chemicals $\rightarrow$ 

Internally uniform the units of Hg and Zn values $\rightarrow$

Record units.

**Output**: a cleaned chemical data set of shape (245, 22), ready for merging.

**Details**:

Specifically using the third sheet in the `log21P_MarjorChe_Sites.xlsx` file, which contains the chemical data mainly collected in 1999, 2004 and 2005.

1. Remove the rows with all missings in all chemical columns, which are the rows with year label - 1991.
Retain only the needed 16 chemical columns with site IDs and relevant information.

1. Back-transform the log2(Y+1) values to the original Y values.
   
2. Infer the units of these chemicals in Y values.
    
    Reasons:
   - There were no attached units in the original dataset.
   - These chemicals were apparently in different units as some chemicals are trivial elements needing smaller scales, while some chemicals are compounds needing larger scales.

3. Inspect the **Hg** and **Zn** columns and internally uniform the units of values within them. 
   
   That is: the 2004/2005 Hg values in **ug/g** are $\rightarrow$ **ng/g**; the 1999 Zn values in **mg/g** are $\rightarrow$ **ug/g**.

   $$\text{Hg}_{ug/g} \times 1000 = \text{Hg}_{ng/g}$$
    $$\text{Zn}_{mg/g} \times 1000 = \text{Zn}_{ug/g}$$
   
   Reasons:
   - The Hg values collected in 2004 and 2005 were in **ug/g** unit, while the rest of the Hg values were in **ng/g** unit. These 2004 and 2005 values are apparently **smaller** than others and perfectly related with the year labels.
 
   - The Zn values collected in 2004 and 2005 were in **ug/g** unit, while the rest values were in **mg/g**. These 2004 and 2005 values are much larger than others and perfectly related with the year labels.

**Conclusion**: After the steps from 0 to 3, the original ``log21P_MarjorChe_Sites.xlsx`` dataset is cleaned with 245 rows of complete chemical data in internally uniformed units. Not all the chemicals are in same units, some are in ug/g and other are in ng/g, this needs to be operated when merging with the `1991Che_DRSites.xlx` dataset.

### Clean the 1991 chemical dataset

**Input**: the originally raw and dirty `1991Che_DRSites.xls` dataset.

**Process**: Retain needed columns $\rightarrow$ Fill missings $\rightarrow$ Record units.

**Output**: the cleaned 1991 chemical data of shape (77, 22) ready for merging.

**Details**:

0. Retain the same 16 targeted chemical columns with site IDs and relevant information.

1. Fill the missing values in Hg(1), SumPCBs(45), Cd(2), OCS(63) and p,p'-DDE(52).
Apply the same filling approach with unifom random values draw from interval [0.01X, 0.5X], where X is the detection limit of each chemical. The detection limits X recorded in the `1991Che_DRSites.xls` data were in consistent units as their reponsible chemicals, not uniformly but internally consistent.

    Reasons:
    - There were missing values in this dataset, specifically 1 missing in Hg, 45 missings in SumPCBs, 2 missings in Cd, 63 missings in OCS and 52 missings in p,p'-DDE. 
    - The filling approach is based on Jian's procedure in 2008, page 31.

2. Record the units of these chemicals in the cleaned 1991 DR dataset.
   
   Reasons:
      - The unit array of the 16 chemicals in this 1991 dataset is not same as the unit array of the 16 chemicals in the cleaned log21 major chemical dataset.
      - In order to merge these two cleaned datasets, their units need to be compared first and then transform them into a uniformed unit array.

**Conclusions**: As the units scales were already clean in the original 1991 dataset, not much inspection work was needed. Retaining the necessary chemical columns and filling the missings makes the 1991 dataset clean and ready for merging. The last and must thing is to record the units of these chemicals in the cleaned 1991 dataset, which will and must guide how to merge it with the cleaned log21 major chemical dataset.

### Merge the two cleaned datasets for a clean and complete chemical dataset

**Inputs**: the cleaned log21 major chemical dataset and the cleaned 1991 DR chemical dataset.

**Process**:

Compare and uniform the units of the two data sets, elementwise separately $\rightarrow$

Scan the cleaned log21 major chemical dataset, drop off the duplicated rows that can be found in the cleaned 1991 dataset $\rightarrow$

Concatenate the two datasets $\rightarrow$

Record the units of the merged dataset

**Outputs**:

A complete and row-wise unique chemical dataset of shape (322, 22), ready for use.

**Details**:

1. Compare the units of the two cleaned datasets, elementwise, use a target unit array to uniform their units.

    Reasons:
    - As did discussed before, the unit arrays of the two cleaned datasets are not same, merging them needs to 
    uniform their units by a universal unit array.

2. Scan the cleaned log21 major chemical dataset, drop off the duplicated rows that can be found in the cleaned 1991 dataset. It resulted 233 sites collected in 1999, 2004 and 2005 in the deduplicated log21 major chemical dataset.

    Reasons:
    - The log21 major chemical dataset contains samples collected across 1991, 1999, 2004 and 2005, but only the 1991 samples are not complete (not having missing but there were more samples collected in 1991).
  
    - The 1991 DR chemical dataset contains all samples collected in 1991, which are complete and therefore
    overlap with the 1991 samples in the log21 major chemical dataset.

2. Concatenate the two datasets.

    Reasons:
    - It is not riching more information around sites by merging data in sample-wise,
    but it is stacking more the same type of information with more samples.
    - Therefore, no common key is needed for concatenating the two datasets, only the same column shape and names are needed.

3. Record the units of the merged dataset.

    Reasons:
    - These units are the first-hand numerical scales of them, reflect the interpretation and 
    almost determine the **effects of transformations** that are applied right before modeling.

**Conclusions**:
After the steps from 0 to 3, the two datasets were stacked together with a total 322 rows of complete chemical data,
16 chemical columns and 6 columns of site IDs and relevant information (322, 16 + 6).

The finalized universal unit array that transformed the two distinct unit arrays is listed below:

<div align="center">

<table style="font-size: 12px; text-align: center;">
  <thead>
    <tr>
      <th>Chemical</th>
      <th>major_set_unit</th>
      <th>1991_DR_unit</th>
      <th>target_unit</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>Co</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr><td>Al</td><td>ug/g</td><td>mg/g</td><td>ug/g</td></tr>
    <tr><td>Ni</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr><td>Mn</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr><td>Fe</td><td>ug/g</td><td>mg/g</td><td>ug/g</td></tr>
    <tr><td>Cr</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr><td>Cu</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr><td>Hg</td><td>ng/g or ug/g</td><td>ug/g</td><td>ng/g</td></tr>
    <tr><td>Pb</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr><td>Zn</td><td>ug/g or mg/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr><td>SumPCBs</td><td>ng/g</td><td>ng/g</td><td>ng/g</td></tr>
    <tr><td>Cd</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr><td>OCS</td><td>ng/g</td><td>ng/g</td><td>ng/g</td></tr>
    <tr><td>p,p'-DDE</td><td>ng/g</td><td>ng/g</td><td>ng/g</td></tr>
    <tr><td>As</td><td>ug/g</td><td>ug/g</td><td>ug/g</td></tr>
    <tr><td>Ca</td><td>ug/g</td><td>mg/g</td><td>ug/g</td></tr>
  </tbody>
</table>

</div>

Notes: 
1. The Hg and Zn values were in mixed units in the original log21 major chemical dataset, so they were uniformed into one consistent unit (ng/g for Hg and ug/g for Zn) first and then converted to the target unit (ng/g for Hg and ug/g for Zn).

2. The transformation between units:
   $$
   1 \text{ mg/g} = 1000 \text{ ug/g}; \quad 1 \text{ ug/g} = 1000 \text{ ng/g}
   $$

# Done in Data Preparation by merging (environmental, taxa, chemical) tabulars

The chemical data takes most time in data preparing, the other two data sets - environmental features and
taxa relative abundances are clean already.

### The common keys for merging the three

The three tabulars can be merged in site-wise(row) manner by two common keys - `StationID` and `Integrated Code`.
The shared index columns are 'StationID' and 'Integrated Code'. '**StationID**' can be found in 
(environmental, chemical) tabulars, while **Integrated Code** can be found in (taxa, chemical) tabulars.

Therefore, a site can have its three types of data merged by the keys - (StationID, Integrated Code) as following:

$$
\text{environmental} \xleftrightarrow{\text{StationID}} \text{chemical} \xleftrightarrow{\text{Integrated Code}} \text{taxa}
$$

Other sample information such as year, waterbody, etc. will be saved only once by complementary searching from the three tabulars, because a site might miss its sample information in one tabular but can be found in the other tabulars.

### The union of sites in all three tabulars (shape of the merged dataset)

Initially, the three tabulars were in the following shapes and units:

<div align="center">

<table style="margin-left: center; margin-right: center; text-align: center;">
    <thead>
        <tr><th>data type</th><th>shape</th><th>unit</th></tr>
    </thead>
    <tbody>
        <tr><td>environmental</td><td>(323, 5 + 6)</td><td>unit array (6, 1)</td></tr>
        <tr><td>taxa</td><td>(311, 5 + 16)</td><td>octave format</td></tr>
        <tr><td>chemical</td><td>(322, 6 + 16)</td><td>unit array (16, 1)</td></tr>
    </tbody>
</table>

</div>

Each row in each tabular can merge with the rows in the other two tabulars by the common keys, but not all rows can find their matches in the other two tabulars. The union of sites that can be found in all three tabulars are the
rows having complete data in all three types. After merging and reviewing, there are 310 sites having complete data in all three types with shape:

<div align="center">

<table style="margin-left: center; margin-right: center; text-align: center;">
  <thead>
    <tr><th>data type</th><th>shape</th></tr>
  </thead>
  <tbody>
    <tr><td>(environmental, chemical, taxa)</td><td>(310, 5 + 6 + 16 + 16)</td></tr>
  </tbody>
</table>

</div>

**Conclusions**:


Complete data of all three types are clean and ready to use, the units of the three types are different and need to explain before analysis. The taxa values were in **log2-transformed relative abundace with a pseudo-count** or
in **octave-scale transformation of relative abundance** (Gauch, 1982) originally.
Its formula is attached below, where $p_ij$ is the proportion of taxa $j$ at site $i$ and the $y_{ij}$ is the
octave-scale transformed value of taxa $j$ at site $i$. 
This octave default values support to infer the relative abundances ($p_{ij}$) of taxa $j$ at site $i$, 
but it is impossible to infer the raw absolute abundances.

$$
y_{ij} = log_{2} [100 (p_{ij} + 0.01)]
$$


Let us call them mathematically as matrices with the following notations:
| Data Matrix Source | Symbol | Dimension | Units |
|---|---|---|---|
| Environmental data | $\mathbf{E}$ | $310 \times 6$ | unit array $(6, 1)$ |
| Chemical data (material composition) | $\mathbf{M}$ | $310 \times 16$ | unit array $(16, 1)$ |
| Taxa data (relative abundances) | $\mathbf{T}$ | $310 \times 16$ | octave format |

where $\mathbf{E}$ denotes environmental data, $\mathbf{M}$ denotes chemical data, and $\mathbf{T}$ denotes taxa data.

# Workflow Template: Recursive Input-Process-Output Framework

The **input-process-output** framework describes the computational logic of each chapter. Every object involved in the workflow should be assigned to exactly one of three roles: input, process, or output.

$$
\text{Inputs} \rightarrow \text{Process} \rightarrow \text{Outputs}
$$

The three object types are defined as follows:

<table>
    <thead>
        <tr>
            <th>Object Type</th>
            <th>Elements</th>
            <th>Where from/to</th>
        </tr>
    </thead>
    <tbody>
        <tr>
            <td rowspan="2">Inputs</td>
            <td>metadata</td>
            <td>retrieved from the complete metadata archive</td>
        </tr>
        <tr>
            <td>artifacts</td>
            <td>numerical artifacts inherited from earlier chapter outputs</td>
        </tr>
        <tr>
            <td>Process</td>
            <td>transformations, bridging tools, models, interpreters</td>
            <td>applied to inputs to generate numerical and visual outputs</td>
        </tr>
        <tr>
            <td rowspan="2">Outputs</td>
            <td>results</td>
            <td>numerical tables or visual figures retained for discussion within the chapter</td>
        </tr>
        <tr>
            <td>artifacts</td>
            <td>results that must be passed forward as inputs to the next chapter</td>
        </tr>
    </tbody>
</table>

For example, if five chapters involve computation and each chapter builds on earlier results, the same framework can be reused recursively across all five stages.

```text
# inputs for stage 0 have already been prepared
For each stage i in [0, 1, 2, 3, 4]:

    # run the chapter-specific process
    inputs -> process.i -> outputs

    # discuss the outputs in chapter i
    outputs -> discussion in chapter i

    # prepare inputs for stage i + 1
    inputs <- (fetch.chapter(i+1).metadata, outputs.artifacts)
```

# Chapter 2: Pollution Assessment and Identification of Least-Polluted Sites

## Framework for Contamination Stressor Assessment

**Question/Motivation**:

Reduce the 16 chemical descriptors to a smaller set of composite variables, interpreted here as contamination stressors. Then convert those stressors into site-level contamination scores that summarize overall contamination and support site ranking.

**Methods and Steps**:

Preprocess the chemical matrix $\mathbf{M}$ so that it is both numerically comparable and ecologically interpretable. Then apply PCA to summarize the 16 chemicals into a smaller number of component loading patterns, and rotate the loadings to improve interpretability in both the loadings and the resulting scores. Finally, combine the retained component scores into a single contamination indicator.

**Inputs**:
1. metadata: chemical data $\mathbf{M}$
2. artifacts: none

**Process**:
1. transformations: $\log_{10}(X + 1)$, z-score standardization, correlation matrix
2. modeling: PCA, quartimax rotation
3. bridging tools: $f_{sum}()$, $f_{max}()$
4. interpreters: rotated loadings, PC scores, and contamination scores derived from $f_{sum}()$ and $f_{max}()$

**Outputs**:
1. results:
   - tables: rotated PC loadings
   - figures: map of sites with $f_{sum}()$ scores; map of sites with $f_{max}()$ scores
2. artifacts:
   - table: PC scores together with $f_{sum}()$ and $f_{max}()$ scores

**Demo Outputs**:

<div style="width: 80%; max-width: 700px; margin: 0 auto;">
  <img src="demos_images/SumRel_corridor_bifurcation.png" alt="SumRel corridor bifurcation" style="width: 100%; display: block;">
  <p style="margin-top: 8px;"><em>Figure: map of lowest 62 (20%, green), middle 183 (60%, grey), and highest 62 (20%, red) sites ranked by SumRel. Empirical Cumulative Distribution Function of the SumRel Scores.</em></p>
</div>

## Framework for the Ranked Subset-RDA Sweep Method

**Question/Motivation**:
Test the assumption that community composition at the least-stressed sites is shaped primarily by habitat features rather than by chemical stress.

**Validation Method**:
Treat the taxa matrix $\mathbf{T} \in \mathbb{R}^{310 \times 16}$ as the response matrix for community composition, and compare how well different types of descriptors explain variation in $\mathbf{T}$.

$$
\mathbf{T} = \mathbf{A}_{\text{descriptors}}\beta + \epsilon
$$

where

$$
\text{descriptors} \in \text{(environmental features, stressors, scores)}
$$

If this assumption is correct, environmental descriptors should explain more variation in $\mathbf{T}$ than the stressor-based descriptors.

**Inputs**:
1. metadata: environmental data $\mathbf{E}$, taxa data $\mathbf{T}$
2. artifacts:
   - table: PC scores together with $f_{sum}()$ and $f_{max}()$ scores

**Process**:
1. transformations: $\ln(X + 1)$, z-score standardization
2. bridging tools: rank-based subset selector, subset selector
3. modeling: Redundancy Analysis (RDA)
4. interpreters: explained inertia, adjusted $R^2$ from RDA, F-statistics from permutation tests, and p-values from permutation tests

**Outputs**:
1. results:
   - tables:
      - summary of RDA sweep results constrained by $f_{sum}$: $(m, \bar{R}^2, \text{F-statistic}, p\text{-value}, \text{cut-off } f_{sum})$, with $m \in [30, 35, 40, \ldots, 310]$
      - summary of RDA sweep results constrained by $f_{max}$: $(m, \bar{R}^2, \text{F-statistic}, p\text{-value}, \text{cut-off } f_{max})$, with $m \in [30, 35, 40, \ldots, 310]$

   - figures:
      - plot of $\bar{R}^2$ for the three descriptor types versus $m$, constrained by $f_{sum}$
      - plot of p-values for the three descriptor types versus $m$, constrained by $f_{max}$
2. artifacts: none


**Demo Outputs**:

<div style="width: 80%; max-width: 700px; margin: 0 auto;">
  <img src="demos_images/SumRel_r2_env_vs_stressor.png" alt="SumRel R2 Environment vs Stressor" style="width: 100%; display: block;">
  <p style="margin-top: 8px;"><em>Figure: The Adjusted R2 values of three types of descriptors passed into RDA to regress taxa matrix with incrementally larger subsets of sites ranked by SumRel.</em></p>
</div>

# Chapter 3: Environmental Standardization of Zoobenthic Community Composition Using Reference-Site Taxa Clusters


This chapter **controls environmental confounding** in zoobenthic community composition through three frameworks. 

First, reference community types are identified from taxa composition at least-stressed sites. Second, their associated environmental conditions are modeled with a classifier. Third, the classifier is applied to all sites to assign each site to an **expected** reference community type based on its environmental descriptor space. 

These steps indirectly partitions environmental space into (K) reference-community regions. Although this discretizes continuous environmental gradients, it improves interpretability with environmentally comparable groups.


## Framework 1: Reference Community Typology from Least-Stressed Sites


To define naturally occurring community types without imposing environmental variables on the clustering process, an **unconstrained** taxa-based clustering was first applied to the least-stressed sites. The resulting reference community types were then used as **response** labels in an environmental classifier.


**Question/Motivation**:

It risks collinearity and loss of interpretability to crunch the environmental covariates together with the stressors into a regression model. Instead, we want the bioassessment idea of comparing observed communities with
expected reference conditions with similar environmental conditions.

Reference communities can have distinct environmental conditions but **are not** defined by them, because contamination can be found in any environmental context. 
Reference communities **should be** described by taxa compositions where contamination is least likely to have shaped community composition, which are the sites with lowest contamination levels.



**Validation Method**:

Pick out the $m$ least-stressed sites based on ranking by the $f_{sum}()$ scores, and apply an unconstrained hierarchical clustering to their taxa matrix $\mathbf{T_{ref}}$. Within the resulting dendrogram, supply statistical evidences to decide how many clusters to retain for cluster **robustness** and **distinctiveness**.




**Inputs**:
1. metadata: taxa data $\mathbf{T}$
2. artifacts:
   - table: PC scores together with $f_{sum}()$ and $f_{max}()$ scores

**Process**:
1. transformations: none, $\mathbf{T}$ was already in octave format
2. bridging tools: rank-based subset selector
3. modeling: Ward's hierarchical clustering, Node-wise ANOVA, ANOVA, pvclust bootstrap resampling, 
4. interpreters: cluster dendrogram with $k$ finalized colors,
node-wise ANOVA results, ANOVA results, approximately unbiased (AU) p-values from pvclust,



**Outputs**:
1. results:
   - tables:
      - Selected $k$ cluster labels for the $m$ least-stressed sites
        - rows: $(\text{Integrated Code}, \text{Cluster Label})$ 
        - shape: $(m, 2)$
      
      - Node-wise ANOVA results for the first $k+1$ bifurcation nodes in the dendrogram
        - rows: $(\text{Splits}, \text{left-size}, \text{right-size}, \text{Taxon}, \text{Left mean},
         \text{Right mean}, \text{F-statistic}, \text{p-value}, \text{FDR p-value})$
        - shape: $((k+1) \times n_{k_\text{sig taxa}}, 9)$

      - ANOVA results for the $k$ clusters
        - rows: $(\text{Taxon}, \text{SS Between(df)}, \text{SS Within(df)}, \text{F-statistic}, \text{p-value},
          \bar{x}_{C_1}, ..., \bar{x}_{C_k})$
        - shape: $(16, 5 + k)$
      
      - pvclust approxiamte unbiased (AU) p-values for $k$ clusters
        - rows: $(\text{Cluster Label}, \text{AU p-value})$
        - shape: $(k, 2)$
   
   - figures:
      - Dendrogram of hierarchical clustering.
        - x-axis: distance between clusters
        - y-axis: site labels and leaves colored by $k$ cluster labels
        - legend: colors for $k$ clusters, AU p-values for $k$ clusters
      - Barplots of the 16 taxa concentrations across the $k$ clusters.
        - x-axis: 16 ticks for 16 taxa, each tick holds $k$ bars for $k$ clusters
        - y-axis: means of relative concentration $p_{jk}$ of each taxon $j$ in each cluster $k$
        - legend: colors for $k$ clusters
        - anotations: asterisks for significant taxa across the three clusters

2. artifacts: 
   - table:
     - Selected $k$ cluster labels for the $m$ least-stressed sites
         - rows: $(\text{Integrated Code}, \text{Cluster Label})$ 
         - shape: $(m, 2)$

**Demo Outputs**:

<div style="width: 50%; max-width: 700px; margin: 0 auto;">
  <img src="demos_images/ward_dendrogram.png" alt="Ward Dendrogram with AUs" style="width: 100%; display: block;">
  <p style="margin-top: 8px;"><em>Figure: Ward Dendrogram with Approximately Unbiased (AU) p-values for the 3 clusters.</em></p>
</div>

## Framework 2: Taxa–Environment Concordance-Based Classifier Training

**If sites are clustered by environmental descriptors, do they tend to group with sites from the same taxa-defined cluster?**


**Background**

Reference cluster labels are defined from unconstrained clustering in taxa space and are used as response classes for training an environmental classifier. However, taxa-defined clusters are not necessarily well separated in environmental space. Some sites assigned to $C_i$ may have environmental profiles closer to sites in $C_j$, suggesting that the measured environmental descriptors cannot fully distinguish the taxa clusters.

If all sites are used equally in classifier training, the model may be forced to learn from sites where taxa labels and environmental features are not concordant. In that case, classifier performance becomes harder to interpret.

**Target**

This framework identifies sites whose cluster membership is supported in both taxa space and environmental space. Using resampling-based consensus clustering, each site receives a taxa-margin and an environment-margin, defined as the difference between its co-assignment frequency with its own cluster and its strongest alternative cluster.

Sites with positive margins in both spaces are treated as **taxa–environment concordant sites**. These sites are used as the primary training set for the environmental classifier because their taxa-defined labels are also supported by environmental similarity.


<div style="width: 80%; max-width: 700px; margin: 0 auto;">
  <img src="demos_images/idea_concordant_sites2.png" alt="Idea Concordant Sites" style="width: 100%; display: block;">
  <p style="margin-top: 8px;"><em>
  Conceptual illustration of taxa–environment concordance for classifier training. 
  
  Left: a taxa-defined cluster $C_i$ may become more dispersed when projected into environmental descriptor space, indicating that not all taxa-cluster members are environmentally coherent. Concordant sites are the intersection of taxa-stable and environment-stable sites within $C_i$.
    
  Right: applying this criterion across all 3 clusters identifies core sites whose taxa membership is supported by environmental similarity. These concordant sites provide the most reliable training cases for modeling the mapping from environmental descriptors to reference community types.</em></p>
</div>

## Question/Motivation

Some reference sites may not be concordant between their taxa-defined cluster labels and their environmental similarity. Therefore, concordant sites need to be identified and used as the primary training set. Using these sites, LDA and MRT are compared, and the better classifier is selected to assign expected reference community types to all sites from their environmental profiles.

## Methods and Steps

Finalized taxa clusters are used as anchor groups to define each site’s siblings and non-siblings. Consensus clustering is run separately in taxa space and environmental space, producing two co-assignment matrices that
quantify the frequency with which each site is co-assigned with its own cluster siblings and with other clusters non-siblings across resampling iterations.

For each site, a margin is calculated as:

$$
\text{margin of site } i_{\in C_j} = \text{co-assignment with siblings}_{\in C_j} - \text{strongest co-assignment with non-siblings}_{\in C_k, k \neq j}
$$

Sites with non-negative margins in both spaces are defined as **taxa–environment concordant sites**. The other margin combinations are kept for interpretation as atypical, ambiguous, or weakly supported cases.

LDA and MRT are trained on the concordant sites. Their performance is compared using resampling-based cross-validation, full-training accuracy, and confusion matrices. The selected classifier is then applied to all sites to assign expected reference community types.



**Inputs**:
1. metadata: environmental data $\mathbf{E}$, taxa data $\mathbf{T}$
2. artifacts:
    - table: selected $k$ cluster labels for the $m$ least-stressed sites
      - rows: $(\text{Integrated Code}, \text{Cluster Label})$
      - shape: $(m, 2)$ 

**Process**:
1. transformations: z-score standardization
2. bridging tools: rank-based subset selector, cross-classification tool (Cartesian product)
3. modeling: Ward's Hierarchical Clustering, Consensus Clustering, Linear Discriminant Analysis (LDA), Multivariate Regression Trees (MRT)
4. interpreters: stratified confusion matrices, confusion matrix by random-split cross-validation


**Outputs**:
1. results:
    - table 1: stratified confusion matrix for LDA trained with concordant sites
      - rows: $(\text{env status}_i \times \text{taxa status}_i, \text{env status}_i \times \text{taxa status}_j,), i, j \in \{1, 2\}$
      - shape: $(2, 2)$
      
      It has confusion matrix as its elements at each place as follows:

        - rows: $(\hat i \in C_1, \hat i \in C_2, \hat i \in C_3) | i \in C_i$
        - shape: $(3, 3)$

    - table 2: stratified confusion matrix for LDA trained with all ref-sites
      - rows: $(\text{env status}_i \times \text{taxa status}_i, \text{env status}_i \times \text{taxa status}_j,), i, j \in \{1, 2\}$
      - shape: $(2, 2)$
      
      It has confusion matrix as its elements at each place as follows:

        - rows: $(\hat i \in C_1, \hat i \in C_2, \hat i \in C_3) | i \in C_i$
        - shape: $(3, 3)$
        
    
    - table 3 : comparison of random-split cv confusion matrices for LDA trained with different sites
      - rows: $(\text{data type} ,\hat i \in C_1, \hat i \in C_2, \hat i \in C_3) | i \in C_i$
      - shape: $(2 \times 3, 3)$

2. artifacts: 
   - model: a finalized classifier trained with concordant sites, ready to partition the environmental space into expected reference community types

**Demo Outputs**:

<div align="center">

<table style="width:92%; border-collapse:collapse; text-align:center; font-size:15px;">
  <thead>
    <tr style="border-top:3px solid black;">
      <th rowspan="2" style="padding:10px;">Environmental<br>strength</th>
      <th rowspan="2" style="padding:10px;">True label</th>
      <th colspan="3" style="padding:10px; border-bottom:1px solid black;">Taxa Strong</th>
      <th colspan="3" style="padding:10px; border-bottom:1px solid black;">Taxa Weak</th>
    </tr>
    <tr style="border-bottom:3px solid black;">
      <th style="padding:10px;">Cluster C<sub>1</sub></th>
      <th style="padding:10px;">Cluster C<sub>2</sub></th>
      <th style="padding:10px;">Cluster C<sub>3</sub></th>
      <th style="padding:10px;">Cluster C<sub>1</sub></th>
      <th style="padding:10px;">Cluster C<sub>2</sub></th>
      <th style="padding:10px;">Cluster C<sub>3</sub></th>
    </tr>
  </thead>

  <tbody>
    <tr>
      <td rowspan="4" style="padding:10px;"><b>Env Strong</b></td>
      <td style="padding:8px;">Cluster C<sub>1</sub></td>
      <td>5</td><td>0</td><td>0</td>
      <td>0</td><td>0</td><td>0</td>
    </tr>
    <tr>
      <td style="padding:8px;">Cluster C<sub>2</sub></td>
      <td>0</td><td>12</td><td>2</td>
      <td>0</td><td>0</td><td>0</td>
    </tr>
    <tr>
      <td style="padding:8px;">Cluster C<sub>3</sub></td>
      <td>0</td><td>1</td><td>14</td>
      <td>0</td><td>0</td><td>1</td>
    </tr>
    <tr style="border-bottom:3px solid black;">
      <td style="padding:8px;"><b>Accuracy</b></td>
      <td colspan="3"><b>91% (n=34)</b></td>
      <td colspan="3"><b>100% (n=1)</b></td>
    </tr>

    <tr>
      <td rowspan="4" style="padding:10px;"><b>Env Weak</b></td>
      <td style="padding:8px;">Cluster C<sub>1</sub></td>
      <td>0</td><td>3</td><td>1</td>
      <td>0</td><td>0</td><td>0</td>
    </tr>
    <tr>
      <td style="padding:8px;">Cluster C<sub>2</sub></td>
      <td>1</td><td>2</td><td>19</td>
      <td>0</td><td>0</td><td>0</td>
    </tr>
    <tr>
      <td style="padding:8px;">Cluster C<sub>3</sub></td>
      <td>1</td><td>0</td><td>0</td>
      <td>0</td><td>0</td><td>0</td>
    </tr>
    <tr style="border-bottom:3px solid black;">
      <td style="padding:8px;"><b>Accuracy</b></td>
      <td colspan="3"><b>7% (n=27)</b></td>
      <td colspan="3"><b>n=0</b></td>
    </tr>
  </tbody>
</table>

</div>